In [134]:
import pandas as pd 
import requests
import io
import time

In [135]:
%pip install lxml

#pd.read_html no se podía ejeecutar porque no tenía instalada la librery lxml.

Note: you may need to restart the kernel to use updated packages.


In [136]:
pd.read_html("https://es.wikipedia.org/wiki/Departamentos_del_Per%C3%BA")

HTTPError: HTTP Error 403: Forbidden

In [137]:
url = "https://es.wikipedia.org/wiki/Departamentos_del_Per%C3%BA"
respuesta = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}) 

tablas = pd.read_html(io.StringIO(respuesta.text))
type(tablas), len(tablas)

(list, 6)

In [138]:
# Seleccionamos las columnas "departamento" y "capital" y las renombramos

df = tablas[1]
df = df[["Departamento", "Capital"]]. copy()
df.columns = ["departamento", "capital"]
print(df)

     departamento            capital
0        Amazonas        Chachapoyas
1          Áncash             Huaraz
2        Apurímac            Abancay
3        Arequipa           Arequipa
4        Ayacucho           Ayacucho
5       Cajamarca          Cajamarca
6           Cusco              Cusco
7    Huancavelica       Huancavelica
8         Huánuco            Huánuco
9             Ica                Ica
10          Junín           Huancayo
11    La Libertad           Trujillo
12     Lambayeque           Chiclayo
13           Lima  Huacho (de facto)
14         Loreto            Iquitos
15  Madre de Dios   Puerto Maldonado
16       Moquegua           Moquegua
17          Pasco     Cerro de Pasco
18          Piura              Piura
19           Puno               Puno
20     San Martín          Moyobamba
21          Tacna              Tacna
22         Tumbes             Tumbes
23        Ucayali           Pucallpa


In [139]:
# Falta el Callao. No aparece en la tabla de departamentos porque es la Provincia Constitucional del Callao, 
# que tiene régimen especial que en la misma página de Wikipedia está en otra tabla, 
# "Provincias de régimen especial" (tablas[3]), con capital Callao. Agregamos manualmente el dato.

if "Callao" not in df["departamento"].values: 
    df.loc[len(df)] = ["Callao", "Callao"]
print(df)

     departamento            capital
0        Amazonas        Chachapoyas
1          Áncash             Huaraz
2        Apurímac            Abancay
3        Arequipa           Arequipa
4        Ayacucho           Ayacucho
5       Cajamarca          Cajamarca
6           Cusco              Cusco
7    Huancavelica       Huancavelica
8         Huánuco            Huánuco
9             Ica                Ica
10          Junín           Huancayo
11    La Libertad           Trujillo
12     Lambayeque           Chiclayo
13           Lima  Huacho (de facto)
14         Loreto            Iquitos
15  Madre de Dios   Puerto Maldonado
16       Moquegua           Moquegua
17          Pasco     Cerro de Pasco
18          Piura              Piura
19           Puno               Puno
20     San Martín          Moyobamba
21          Tacna              Tacna
22         Tumbes             Tumbes
23        Ucayali           Pucallpa
24         Callao             Callao


In [140]:
# Si bien Huacho es la capital de Lima región, se considera Lima Metropolitana como "capital" 
# para facilitar el análisis.

df.loc[df["departamento"] == "Lima", "capital"] = "Lima"
print(df)

     departamento           capital
0        Amazonas       Chachapoyas
1          Áncash            Huaraz
2        Apurímac           Abancay
3        Arequipa          Arequipa
4        Ayacucho          Ayacucho
5       Cajamarca         Cajamarca
6           Cusco             Cusco
7    Huancavelica      Huancavelica
8         Huánuco           Huánuco
9             Ica               Ica
10          Junín          Huancayo
11    La Libertad          Trujillo
12     Lambayeque          Chiclayo
13           Lima              Lima
14         Loreto           Iquitos
15  Madre de Dios  Puerto Maldonado
16       Moquegua          Moquegua
17          Pasco    Cerro de Pasco
18          Piura             Piura
19           Puno              Puno
20     San Martín         Moyobamba
21          Tacna             Tacna
22         Tumbes            Tumbes
23        Ucayali          Pucallpa
24         Callao            Callao


In [141]:
# Definimos la API de geocoding como variable. Y la variable "coordenadas" con la indicación de búsqueda:
# para cada capital en df, buscar la primera capital en Perú que aparezca en geocoding y generar una tabla con capital,
# latitud y longitud. 

url_geo = "https://geocoding-api.open-meteo.com/v1/search"
coordenadas = []

for capital in df["capital"]:
    parametros = {
        "name": capital,
        "count": 1,
        "language": "es",
        "countryCode": "PE",
    }

    respuesta = requests.get(url_geo, params=parametros)
    datos = respuesta.json()
    r = datos["results"][0]
    coordenadas.append({"capital": capital, "latitud": r["latitude"], "longitud": r["longitude"]})

len(coordenadas)

25

In [142]:
# Combina ambos dataframes teniendo a "capital" como variable unificadora

df_coord = pd.DataFrame(coordenadas)
df = df.merge(df_coord, on="capital", how="left")
print(df)

     departamento           capital   latitud  longitud
0        Amazonas       Chachapoyas  -6.23169 -77.86903
1          Áncash            Huaraz  -9.52614 -77.52869
2        Apurímac           Abancay -13.63426 -72.88380
3        Arequipa          Arequipa -16.39899 -71.53747
4        Ayacucho          Ayacucho -13.16380 -74.22345
5       Cajamarca         Cajamarca  -7.16378 -78.50027
6           Cusco             Cusco -13.53188 -71.96701
7    Huancavelica      Huancavelica -12.78691 -74.97298
8         Huánuco           Huánuco  -9.92882 -76.23989
9             Ica               Ica -14.07538 -75.73422
10          Junín          Huancayo -12.06866 -75.21027
11    La Libertad          Trujillo  -8.11599 -79.02998
12     Lambayeque          Chiclayo  -6.77008 -79.85495
13           Lima              Lima -12.04318 -77.02824
14         Loreto           Iquitos  -3.74814 -73.25290
15  Madre de Dios  Puerto Maldonado -12.58946 -69.19948
16       Moquegua          Moquegua -17.19747 -7

In [ ]:
# Extraemos la información del API de lluvias en una lista vacía denominada lluvias. 

url_clima = "https://archive-api.open-meteo.com/v1/archive"
lluvias = []

# Define que recorra el df fila por fila y estable los parámetros del pedido de búsqueda. 
 
for i, fila in df.iterrows():
    parametros = {
        "latitude": fila["latitud"],
        "longitude": fila["longitud"],
        "start_date": "2022-01-01",
        "end_date": "2022-05-31",
        "daily": "precipitation_sum",   
        "timezone": "America/Lima",
    }
    respuesta = requests.get(url_clima, params=parametros)
    datos = respuesta.json()
    
    # Si aparece el resultado "daily" es una búsqueda exitosa y arma la información, de lo contrario arroja un error. 
    
    if "daily" in datos:
        temp = pd.DataFrame({
            "fecha": datos["daily"]["time"],
            "lluvia_mm": datos["daily"]["precipitation_sum"],
        })
        temp["departamento"] = fila["departamento"]
        temp["capital"] = fila["capital"]
        lluvias.append(temp)          # ← esto es lo que llena la lista
    else:
        print("Error en", fila["capital"], datos)

    time.sleep(1) # Tiempo entre búsquedas de 1 segundo.

# Une todas las búsquedas en una sola lista 

df_lluvia = pd.concat(lluvias, ignore_index=True)
df_lluvia = df_lluvia[["departamento", "capital", "fecha", "lluvia_mm"]]

print(df_lluvia.head())
print(len(df_lluvia))
print(df_lluvia["capital"].nunique())


  departamento      capital       fecha  lluvia_mm
0     Amazonas  Chachapoyas  2022-01-01        3.1
1     Amazonas  Chachapoyas  2022-01-02        3.8
2     Amazonas  Chachapoyas  2022-01-03        0.9
3     Amazonas  Chachapoyas  2022-01-04        2.5
4     Amazonas  Chachapoyas  2022-01-05        1.2
3775
25


In [ ]:
# Se crea una variable que agrupa todos los valores de lluvia_mm por departamento. 
# Al aplicar groupby se crea un dataframe. 

lluvia_total_mm = (
    df_lluvia.groupby("departamento", as_index=False)["lluvia_mm"]
    .sum()
    .rename(columns={"lluvia_mm": "lluvia_total_mm"})
    .round(1)
)

print(lluvia_total_mm)
print(len(lluvia_total_mm))


     departamento  lluvia_total_mm
0        Amazonas            436.5
1        Apurímac            711.8
2        Arequipa            189.9
3        Ayacucho            485.1
4       Cajamarca            773.3
5          Callao             39.2
6           Cusco            710.7
7    Huancavelica            803.8
8         Huánuco            444.8
9             Ica             25.1
10          Junín            910.4
11    La Libertad             98.9
12     Lambayeque             98.8
13           Lima             39.2
14         Loreto           1581.8
15  Madre de Dios            908.0
16       Moquegua             17.8
17          Pasco            774.0
18          Piura             89.2
19           Puno            779.5
20     San Martín            789.8
21          Tacna             11.5
22         Tumbes            332.3
23        Ucayali           1288.8
24         Áncash            975.3
25


In [ ]:
# Se crea una variable donde se le asigna la condición de días con lluvia mayor o igual a 20mm. 
# Se agrupan por departamento todos los días con mayor o igual a 20 mm y se suman. 
# Se le cambia el nombre al nombre de la variable.

dias_lluvia_fuerte = (
    df_lluvia.assign(dia_20mm=df_lluvia["lluvia_mm"] >= 20)
    .groupby("departamento", as_index=False)["dia_20mm"]
    .sum()
    .rename(columns={"dia_20mm": "dias_lluvia_fuerte"})
)

print(dias_lluvia_fuerte)
print(len(dias_lluvia_fuerte))

     departamento  dias_lluvia_fuerte
0        Amazonas                   1
1        Apurímac                   2
2        Arequipa                   0
3        Ayacucho                   0
4       Cajamarca                   1
5          Callao                   0
6           Cusco                   3
7    Huancavelica                   1
8         Huánuco                   1
9             Ica                   0
10          Junín                   5
11    La Libertad                   1
12     Lambayeque                   0
13           Lima                   0
14         Loreto                  25
15  Madre de Dios                  13
16       Moquegua                   0
17          Pasco                   0
18          Piura                   0
19           Puno                   3
20     San Martín                   4
21          Tacna                   0
22         Tumbes                   1
23        Ucayali                  20
24         Áncash                   7
25


In [ ]:
# Agrupamos el df base, con los nuevos df que generamos en las líneas anteriores. 
# Partimos de las columnas base para que la celda se pueda ejecutar varias veces

df = df[["departamento", "capital", "latitud", "longitud"]]

df = df.merge(lluvia_total_mm, on="departamento", how="left")
df = df.merge(dias_lluvia_fuerte, on="departamento", how="left")

print(df)
print(df.columns.tolist())
print(len(df))


     departamento           capital   latitud  longitud  lluvia_total_mm  \
0        Amazonas       Chachapoyas  -6.23169 -77.86903            436.5   
1          Áncash            Huaraz  -9.52614 -77.52869            975.3   
2        Apurímac           Abancay -13.63426 -72.88380            711.8   
3        Arequipa          Arequipa -16.39899 -71.53747            189.9   
4        Ayacucho          Ayacucho -13.16380 -74.22345            485.1   
5       Cajamarca         Cajamarca  -7.16378 -78.50027            773.3   
6           Cusco             Cusco -13.53188 -71.96701            710.7   
7    Huancavelica      Huancavelica -12.78691 -74.97298            803.8   
8         Huánuco           Huánuco  -9.92882 -76.23989            444.8   
9             Ica               Ica -14.07538 -75.73422             25.1   
10          Junín          Huancayo -12.06866 -75.21027            910.4   
11    La Libertad          Trujillo  -8.11599 -79.02998             98.9   
12     Lamba

In [ ]:
# Se exporta el df final en la carpeta datos, en formato csv, retirando los números de las columnas, 
# definiendo el punto y coma como separador, el punto como separador decimal y el enconding. 

df.to_csv("datos/lluvias_por_departamento.csv", index=False, sep=";", decimal=".", encoding="utf-8-sig")